# Synthetic claims journeys and time-to-event methods

This notebook demonstrates deterministic claims engineering and Kaplan–Meier calculations on **CMS DE-SynPUF 2008–2010 public synthetic data**. The data are synthetic Medicare-like records for method demonstration only; results are **not representative of Medicare beneficiaries** and no estimate is generalized to Medicare.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from ckd_intelligence.journeys.synpuf import build_journey_output, validate_journey_order
from ckd_intelligence.statistics.time_to_event import kaplan_meier

fixture = ROOT / 'data' / 'fixtures' / 'synpuf_journeys.csv'
claims = pd.read_csv(fixture)
output = build_journey_output(claims)
validate_journey_order(output.events)
assert set(output.events['evidence_type']) == {'public_synthetic'}
print({'evidence_type': 'public_synthetic', 'source': 'CMS DE-SynPUF 2008-2010', 'rows': len(claims)})
print(output.rules.rule_text)

## Journey rules

The index date is the first valid claim date per synthetic beneficiary. The target is the first CKD ICD-9 signal. A persistence event requires a second CKD signal at least 90 days later, including the 90-day boundary. A person without a persistence event is right-censored at the earlier of the 365-day follow-up end or the CMS 2010 observation end. These are software-method rules, not clinical definitions.

In [ ]:
print('Synthetic journey events:')
display(output.events)
print('Synthetic beneficiary summaries:')
display(output.summaries)

In [ ]:
km = kaplan_meier(output.summaries['duration_days'], output.summaries['event_observed'])
display(km.table)
print({'median_survival_days': km.median_survival, 'confidence_level': km.confidence_level})
assert (km.table['survival'].diff().dropna() <= 0).all()
print('Interpretation boundary: this Kaplan–Meier table describes only these CMS synthetic records and is not representative of Medicare beneficiaries.')

## Limitations

CMS DE-SynPUF is a public synthetic sample. It does not support claims about real Medicare utilization, persistence, incidence, treatment, outcomes, or patient journeys. The notebook intentionally keeps this output separate from all public-observed NHANES, MEPS, Part D, and ClinicalTrials.gov analyses.